In [1]:
!pip install diffusers transformers accelerate safetensors controlnet_aux


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 17.6 MB/s eta 0:00:00


In [3]:
import requests
from PIL import Image

# Enlace a una imagen de arquitectura ideal para Canny (Unsplash)
url = "https://images.unsplash.com/photo-1600585154340-be6161a56a0c?q=80&w=512&h=512&fit=crop"
image = Image.open(requests.get(url, stream=True).raw)

print("Imagen base 'image' cargada correctamente.")

Imagen base 'image' cargada correctamente.


In [5]:
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from controlnet_aux import CannyDetector

# 1. Cargar modelo de ControlNet especificando explícitamente el tipo de dato float16
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny",
    torch_dtype=torch.float16 # <--- ESTO ARREGLA EL ERROR
)

# Cargar el pipeline base amarrado al controlnet
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to("cuda")

# 3. Aplicar el detector de condición (Canny)
detector = CannyDetector()
condition_image = detector(image)

# Guardar la imagen de condición para tu carpeta media/
condition_image.save("condicion_canny.png")

# 4. Generar imagen condicionada con prompt
result = pipe(
    "A cyberpunk city skyline at night",
    image=condition_image,
    num_inference_steps=30
).images[0]

# Guardar resultado final
result.save("resultado_controlnet.png")

print("¡Proceso completado con éxito! Las dtypes ahora coinciden perfectamente.")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/30 [00:00<?, ?it/s]

¡Proceso completado con éxito! Las dtypes ahora coinciden perfectamente.


In [6]:
from diffusers import StableDiffusionPipeline

# Liberamos memoria del pipeline anterior para evitar congelar Colab
del pipe
torch.cuda.empty_cache()

# Cargamos el pipeline clásico (solo texto)
pipe_solo_texto = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to("cuda")

# Generamos exactamente el mismo prompt pero sin la guía estructural de la imagen
result_sin_control = pipe_solo_texto("A cyberpunk city skyline at night", num_inference_steps=30).images[0]
result_sin_control.save("resultado_solo_texto.png")

print("Imagen comparativa (sin ControlNet) guardada con éxito.")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/30 [00:00<?, ?it/s]

Imagen comparativa (sin ControlNet) guardada con éxito.


In [7]:
from controlnet_aux import MidasDetector

# 1. Extraer el mapa de profundidad de tu misma imagen base
midas_detector = MidasDetector.from_pretrained("lllyasviel/ControlNet")
condition_depth = midas_detector(image)
condition_depth.save("condicion_depth.png")

# 2. Cargar el modelo de ControlNet Depth en float16
controlnet_depth = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=torch.float16
)

pipe_depth = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet_depth,
    torch_dtype=torch.float16
).to("cuda")

# 3. Generar la imagen con el mismo prompt para comparar
result_depth = pipe_depth(
    "A cyberpunk city skyline at night",
    image=condition_depth,
    num_inference_steps=30
).images[0]

result_depth.save("resultado_depth.png")
print("¡Variación de profundidad (Depth) guardada con éxito!")

annotator/ckpts/dpt_hybrid-midas-501f0c7(…):   0%|          | 0.00/493M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name vit_base_resnet50_384 to current vit_base_r50_s16_384.orig_in21k_ft_in1k.
  model = create_fn(


config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

...

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/30 [00:00<?, ?it/s]

¡Variación de profundidad (Depth) guardada con éxito!
